---
**Central Limit Theorem & Confidence Intervals in Python**
Data Analysis Course · Week 6
---

This notebook is the Python equivalent of the R Markdown `_05_Distributions_CLT.Rmd`.
Topics: the **Central Limit Theorem** (CLT), **standard error**, and **confidence intervals**.

Work through it cell by cell — run each code cell with **Shift+Enter**.

**Required packages:** `numpy`, `pandas`, `matplotlib`, `scipy`
```
pip install numpy pandas matplotlib scipy
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(123)

## 0 – Recap of the previous sheet

Last time we learned about probability distributions and R's/scipy's `p`/`q`/`d`/`r`-style
functions, plus the Normal and Poisson distributions.

## Introduction and objectives

Here we complete our training on distributions with the **Central Limit Theorem** (CLT) and
**confidence intervals**.

## Part 0: an introductory example

Imagine 2 independent random variables. We sample 10 000 points for each.

In [ ]:
x1 = np.random.uniform(size=10000)
x2 = np.random.uniform(size=10000)

In [ ]:
plt.scatter(x1, x2, s=2)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(x1)
axes[1].hist(x2)
plt.show()

Each follows a uniform distribution. Now let's plot the distribution of the *mean* of both variables:

In [ ]:
plt.hist(0.5 * (x1 + x2))
plt.show()

# This is no longer a uniform distribution!
# Try adding a third, fourth, ... variable and repeating. What do you observe?
# How could you show that x1 and x2 are independent?

## Part 1: Central Limit Theorem

We'll use a coffee shop as our running example. First, waiting times at the coffee shop: the owner
wants to know the average waiting time, computed over groups of customers of increasing size.

Individual waiting times are **right-skewed** (most customers wait ~3 minutes, some much longer).
We'll see how the distribution of **sample means** becomes approximately normal even though
individual waiting times are not.

### 1.1 – The original distribution

We model individual waiting times using a **Gamma distribution** (the exact distribution doesn't
matter here — it's just a convenient way to get a right-skewed shape).

- **Mean (μ) = shape × scale = 6 minutes**
- **Standard deviation (σ) = √(shape × scale²) ≈ 4.90 minutes**

*Most of the code below is just for producing the plots — focus on interpreting the visuals, not on understanding every line.*

In [ ]:
# Parameters of the Gamma distribution — not important to know or understand
shape_param = 1.5
scale_param = 4.0

mu = shape_param * scale_param
sigma = np.sqrt(shape_param * scale_param**2)

# scipy's gamma uses the same shape/scale parameterization as R's rgamma(shape=, scale=)
individual_times = stats.gamma.rvs(a=shape_param, scale=scale_param, size=5000)

plt.hist(individual_times, bins=50, density=True, color="steelblue", edgecolor="white")
plt.axvline(mu, color="darkgreen", linestyle="--", linewidth=2)
plt.xlim(0, 30)
plt.xlabel("Waiting Time (minutes)")
plt.ylabel("Density")
plt.title("Individual Waiting Times (Gamma Distribution)")
plt.show()

# Key observation: the distribution is NOT normal — clearly right-skewed!

### 1.2 – The Central Limit Theorem in action

Now we take many samples of **n customers** and compute the average waiting time for each sample.

**CHANGE `number_clients` TO EXPERIMENT!**

In [ ]:
# ============================================
number_clients = 2   # Try: 2, 5, 10, 30, 50, 200
# ============================================

num_days = 1000   # Number of days the survey is conducted

sample_means = np.array([
    stats.gamma.rvs(a=shape_param, scale=scale_param, size=number_clients).mean()
    for _ in range(num_days)
])

theoretical_mean = mu
theoretical_se = sigma / np.sqrt(number_clients)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 10))

# Panel 1: histogram with normal overlay
axes[0].hist(sample_means, bins=40, density=True, color="coral", edgecolor="white")
x_vals = np.linspace(2, 15, 500)
axes[0].plot(x_vals, stats.norm.pdf(x_vals, loc=theoretical_mean, scale=theoretical_se), color="blue", linewidth=2)
axes[0].axvline(theoretical_mean, color="darkgreen", linestyle="--", linewidth=2)
axes[0].set_xlim(2, 15)
axes[0].set_title(f"Distribution of Sample Means (n = {number_clients})")
axes[0].set_xlabel("Sample Mean Waiting Time (minutes)")
axes[0].set_ylabel("Density")

# Panel 2: Q-Q plot
stats.probplot(sample_means, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q Plot: Are Sample Means Normal?")

plt.tight_layout()
plt.show()

### Q-Q plots: assessing normality across sample sizes

In [ ]:
qqplot_sizes = [2, 10, 200]
qqplot_means_list = [
    np.array([stats.gamma.rvs(a=shape_param, scale=scale_param, size=n).mean() for _ in range(5000)])
    for n in qqplot_sizes
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, n, means in zip(axes, qqplot_sizes, qqplot_means_list):
    stats.probplot(means, dist="norm", plot=ax)
    ax.set_title(f"Q-Q Plot: n = {n}")
plt.tight_layout()
plt.show()

**Interpretation:**

- **n = 2:** Points deviate from the line, especially in the tails (still skewed)
- **n = 10:** Much better fit, slight deviations remain
- **n = 200:** Nearly perfect alignment with the theoretical normal line!

**Theoretical Normal Distribution:** the CLT states that sample means follow approximately

$$\bar{X} \sim N\left(\mu, \frac{\sigma^2}{n}\right)$$

### 1.3 – Standard error and sample size

The **standard error** (SE) measures the variability of sample means:

$$SE = \frac{\sigma}{\sqrt{n}}$$

Let's see how SE decreases as sample size increases:

In [ ]:
sample_sizes = np.arange(2, 201, 2)

simulated_se = np.array([
    np.std([stats.gamma.rvs(a=shape_param, scale=scale_param, size=n).mean() for _ in range(1000)], ddof=1)
    for n in sample_sizes
])

theoretical_se_values = sigma / np.sqrt(sample_sizes)

plt.scatter(sample_sizes, simulated_se, color="coral", s=20)
plt.plot(sample_sizes, theoretical_se_values, color="blue", linewidth=2.5)
plt.text(140, max(simulated_se) * 0.92, r"$SE = \sigma / \sqrt{n}$", color="blue", fontsize=12)
plt.grid(color="gainsboro", linestyle="--")
plt.xlabel("Sample Size (n)")
plt.ylabel("Standard Error (SE)")
plt.title("Standard Error vs. Sample Size")
plt.show()

**Key insights:**

1. **Larger samples → smaller SE:** the spread of sample means decreases with √n
2. **Simulated values match theory:** confirms $SE = \sigma/\sqrt{n}$
3. **Diminishing returns:** going from n=10 to n=20 helps more than n=80 to n=90

Try re-running the simulation above with `number_clients` = 2, 5, 30, 100 and compare.

### *Think about it and test*
> 1. At what sample size does the distribution of sample means start looking approximately normal?
> 2. How does the Q-Q plot change as n increases?
> 3. Why does the standard error decrease as sample size increases?
> 4. Does the **mean** of the sample means change with sample size?

The **Central Limit Theorem** tells us: **regardless of the original population distribution**
(even skewed, like our waiting times), the distribution of sample means becomes approximately
normal for a sufficiently large sample size — as long as the random variables are **independent**.

In [ ]:
# Redo the introductory example with dependent variables:
x1 = np.random.uniform(size=10000)
x2 = 1 - x1
# Does the CLT still hold here?

## Part 2: Confidence intervals

The **confidence interval** (CI) describes an interval containing the (unknown) expectation value
of a distribution with, e.g., 95% confidence: out of 100 random realizations, the true expectation
$\mu$ will fall inside this interval about 95 times.

We simulate a random variable following a Poisson distribution:

$$P(x) = \frac{e^{-\lambda}\lambda^x}{x!}$$

Here *we know the true expectation value* — we want to estimate $\lambda$ and check whether our CI
captures it.

### 2.1 – The scenario

A coffee shop manager wants to estimate the average number of customers arriving per 10-minute
window. The (unknown, in practice) true rate is **λ = 50 customers per 10-minute window**.

In [ ]:
########### THE CODE HERE IS NOT IMPORTANT ###########
lambda_ = 50

x_vals = np.arange(25, 76)
probs = stats.poisson.pmf(x_vals, mu=lambda_)

plt.bar(x_vals, probs, color="blue")
plt.xlabel("Number of Customers per 10-minute Window")
plt.ylabel("Probability")
plt.title("Distribution of Customer Arrivals (Poisson, λ = 50)")
plt.show()

# Key property of Poisson: Mean = Variance = λ

### 2.2 – Constructing a confidence interval

#### Example with n = 5

The manager observes 5 separate 10-minute windows per day, counts customers, and averages — repeated
over 100 days.

In [ ]:
lambda_ = 50
n = 5
X = [stats.poisson.rvs(mu=lambda_, size=n) for _ in range(100)]

In [ ]:
Xm = np.array([x.mean() for x in X])     # sample means
Xsd = np.array([x.std(ddof=1) for x in X])  # sample standard deviations

Next, the 95% CI bounds — based on the $t$-distribution with $N-1$ degrees of freedom:

In [ ]:
df = n - 1
tc = stats.t.ppf(0.975, df)   # critical value for df = N-1 and 95% CI

Xl = Xm - tc * Xsd / np.sqrt(n)   # lower bound
Xh = Xm + tc * Xsd / np.sqrt(n)   # upper bound

In [ ]:
i_ok = (Xl < lambda_) & (Xh > lambda_)   # is the true lambda inside the interval?

fig, ax = plt.subplots(figsize=(10, 5))
colors = np.where(i_ok, "blue", "red")
for i in range(len(Xm)):
    ax.plot([i, i], [Xl[i], Xh[i]], color=colors[i], linewidth=2)
ax.scatter(range(len(Xm)), Xm, color="black", s=15, zorder=3)
ax.axhline(lambda_, linestyle=":", color="black")
ax.set_ylim(20, 80)
ax.set_title(f"Mean values and confidence intervals, N={n}")
plt.show()

print("Times the true value was outside the CI:", (~i_ok).sum())
# This should be close to the expected 5%.

The red/blue bars are the confidence interval, the black dot is the sample mean, and the dotted
line at λ is the true expectation value. Blue = CI contains λ, red = it doesn't. How often is the
true value outside the CI? Count the red bars!

### *Think about it and test*
> 1. Repeat with samples of N=24 (100 times). What do you observe?
> 2. How often is λ outside the CI? Try a 90% CI instead — does it still work?
> 3. Why do some intervals miss λ even with a "95% confidence" method?
> 4. Compare n=25 vs. n=100. How much narrower is the CI at n=100?
> 5. What would happen with 99% confidence instead of 95%?

---
## Exercises

### Exercise 1

Repeat the CLT exercise, but starting from a **uniform** distribution: assume customers have
individual waiting times uniform between 0 and 20 minutes. Use `stats.uniform` or `np.random.uniform`
(check the docs!).

In [ ]:
# Your code here:

### Exercise 2

A coffee shop collects customer satisfaction scores (1–10). The manager believes the true average
is around 7.5 and wants to estimate it more precisely with confidence intervals.

1. Generate a sample of n=20 satisfaction scores from a normal distribution with mean=7.5, sd=1.2
   (use `stats.norm.rvs` or `np.random.normal`).
2. Calculate the sample mean and standard deviation.
3. Construct a 95% CI for the true average using the t-distribution.
4. Repeat steps 1–3 100 times to get 100 different CIs.
5. Plot all 100 CIs (like the Poisson example above). Color blue if the CI contains the true mean
   (7.5), red otherwise.
6. Count how many of the 100 intervals contain the true mean. Is it close to 95%?

In [ ]:
# Your code here:

## Summary: What have we learned?

| R | Python | Purpose |
|---|--------|---------|
| `runif(n)` | `np.random.uniform(size=n)` | Uniform random draws |
| `rgamma(n, shape, scale)` | `stats.gamma.rvs(a=shape, scale=scale, size=n)` | Gamma random draws |
| `replicate(n, expr)` | list comprehension / `for` loop | Repeat a simulation |
| `sapply(x, f)` | list comprehension / `np.array([...])` | Apply over a vector |
| `sd(x)` | `np.std(x, ddof=1)` | Sample standard deviation |
| `qt(p, df)` | `stats.t.ppf(p, df)` | t-distribution quantile |
| `rpois(n, lambda)` | `stats.poisson.rvs(mu=lambda, size=n)` | Poisson random draws |
| `dpois(x, lambda)` | `stats.poisson.pmf(x, mu=lambda)` | Poisson probability mass |
| `qqnorm()` + `qqline()` | `scipy.stats.probplot(x, dist="norm", plot=ax)` | Q-Q plot |